# ITRI626 Practical Project: Deep Learning for Image Classification
## Microcontroller & Electronic Component Classification

**Student / Group**: Carlo Barnardo
**Lecturer**: Prof. Absalom Ezugwu  
**Facilitators**: Mr. Bernard Swanepoel  

---
### Project Overview & Objectives
This notebook implements an end-to-end deep learning workflow for classifying microcontroller development boards (Arduino Uno, Arduino Nano, ESP32, ESP8266, Raspberry Pi Pico, STM32 Blue Pill) and electronic components (IC chips, resistors, capacitors, LEDs).

In accordance with the **ITRI626 Rubric**, this project:
1. Prepares, cleans, and splits an image dataset with rigorous **data leakage prevention**.
2. Implements and compares **three distinct model families**:
   - **Family 1**: A custom Convolutional Neural Network baseline (trained from scratch).
   - **Family 2**: Residual Network (`ResNet-18`) leveraging transfer learning with ImageNet pre-trained weights.
   - **Family 3**: Modern inverted residual network (`MobileNetV3-Large`) with pre-trained weights.
3. Evaluates all models on a held-out test split using **Test Accuracy, Macro F1, Precision, Recall, Confusion Matrices, and Loss Curves**.
4. Performs a detailed **qualitative error analysis** on misclassified samples.

### 1. Environment & Hardware Verification

In [ ]:
import sys
from pathlib import Path

# Ensure project src is in Python path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import torchvision
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python Version:      {sys.version.split()[0]}")
print(f"PyTorch Version:     {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"CUDA Available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:          {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 2. Dataset Ingestion, Cleaning & Leakage-Free Splitting
The dataset is divided into **70% Training, 15% Validation, and 15% Test** using stratified sampling with fixed random seed `42`.

In [ ]:
from mlproject.config import default_config
from mlproject.data.download_and_prep import prepare_dataset, print_split_summary

# Ensure data and splits exist (generates demo data if raw is empty)
prepare_dataset(config=default_config, generate_if_empty=True)
print_split_summary(default_config.splits_dir)

### 3. Data Augmentation & DataLoader Verification
Augmentations are **strictly applied to the training set only** (random rotation, flips, color jitter, affine) to prevent data leakage.

In [ ]:
from mlproject.dataset import create_dataloaders

train_loader, val_loader, test_loader, id_to_label, class_weights = create_dataloaders(
    splits_dir=default_config.splits_dir,
    batch_size=16,
    config=default_config,
)

print(f"Classes: {id_to_label}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")
if class_weights is not None:
    print(f"Calculated Class Weights for Loss: {class_weights.numpy().round(3)}")

### 4. Training Model Family 1: Baseline Custom CNN (Trained from Scratch)
A 4-stage convolutional neural network initialized randomly without pre-trained weights.

In [ ]:
from mlproject.train import train_model

cnn_history = train_model(
    model_name="custom_cnn",
    epochs=15,
    batch_size=32,
    lr=1e-3,
    seed=42,
    config=default_config,
)

### 5. Training Model Family 2: ResNet-18 (Transfer Learning)
Residual network initialized with ImageNet pre-trained weights.

In [ ]:
resnet_history = train_model(
    model_name="resnet18",
    epochs=12,
    batch_size=32,
    lr=3e-4,
    seed=42,
    config=default_config,
)

### 6. Training Model Family 3: MobileNetV3-Large (Modern Inverted Residual Architecture)
Lightweight modern architecture with Squeeze-and-Excitation attention.

In [ ]:
mobilenet_history = train_model(
    model_name="mobilenet_v3",
    epochs=12,
    batch_size=32,
    lr=3e-4,
    seed=42,
    config=default_config,
)

### 7. Evaluation & Comparison on Held-Out Test Set
Evaluating test accuracy, macro F1, confusion matrices, and inference latency.

In [ ]:
from mlproject.evaluate import evaluate_model
from mlproject.visualize import plot_learning_curves, plot_model_comparison

models = ["custom_cnn", "resnet18", "mobilenet_v3"]
results = {}
for m in models:
    results[m] = evaluate_model(model_name=m, config=default_config)
    hist_file = default_config.models_dir / m / "history.json"
    if hist_file.exists():
        plot_learning_curves(hist_file, config=default_config)

plot_model_comparison(config=default_config)

### 8. Qualitative Error Analysis
Inspecting misclassified samples to understand failure modes.

In [ ]:
from mlproject.visualize import plot_misclassified_samples

for m in models:
    res_file = default_config.results_dir / f"{m}_results.json"
    if res_file.exists():
        plot_misclassified_samples(res_file, config=default_config)